# Résumé → chunks → embeddings

Validate `resume.yaml`, flatten it into chunks (`data/resume_bullets.json`),
and embed those chunks on the GPU (cached to `cache/resume_embeddings.npz`).
All logic lives in `src/resume.py`; this notebook just drives it.

Edit `resume.yaml`, then run the cells top to bottom.


## Setup: make the project's `src/` importable


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from resume import (
    CACHE,
    OUTPUT_PATH,
    RESUME_PATH,
    flatten,
    load_or_embed,
    load_resume,
    save_chunks,
)

print(f"Reading:  {RESUME_PATH.relative_to(ROOT)}")
print(f"Chunks:   {OUTPUT_PATH.relative_to(ROOT)}")
print(f"Vectors:  {CACHE.relative_to(ROOT)}")

Reading:  resume.yaml
Chunks:   data/resume_bullets.json
Vectors:  cache/resume_embeddings.npz


## Chunk

`load_resume` strictly validates `resume.yaml` (unknown keys fail with a clear
error). `flatten` turns it into ordered chunks; `save_chunks` writes the JSON.


In [2]:
resume = load_resume(RESUME_PATH)
chunks = flatten(resume)
save_chunks(chunks, OUTPUT_PATH)
print(f"Wrote {len(chunks)} chunks -> {OUTPUT_PATH.relative_to(ROOT)}")

Wrote 26 chunks -> data/resume_bullets.json


## Preview the chunks

Each chunk's id, section, and the text that will be embedded.


In [3]:
import pandas as pd

pd.DataFrame(
    {
        "id": [c.id for c in chunks],
        "section": [c.section for c in chunks],
        "text": [c.text for c in chunks],
    }
)

,id,section,text
0,r001,summary,"Engineering leader with 9 years of experience,..."
1,r002,summary,Built and led platform and backend teams at hi...
2,r003,summary,Track record of growing engineers and shipping...
3,r004,experience,Lead a team of 8 backend and platform engineer...
4,r005,experience,Own the reliability and scalability of a multi...
5,r006,experience,Grew the team from 3 to 8 engineers and promot...
6,r007,experience,Partner with product and design on quarterly r...
7,r008,experience,Introduced an on-call rotation and incident re...
8,r009,experience,Designed and shipped a distributed job-process...
9,r010,experience,Mentored four junior engineers through code re...


## Embed on the GPU

`load_or_embed` embeds each chunk with BAAI/bge-large-en-v1.5 on the GPU and
caches the result, keyed by a hash of the chunks file + model name. The first
run loads the model (a few seconds); re-runs are an instant cache hit until
`resume.yaml` changes.


In [4]:
ids, vecs = load_or_embed()
print(f"{len(ids)} embeddings, shape {vecs.shape}, dtype {vecs.dtype}")

Cache hit — 26 embeddings from cache/resume_embeddings.npz
26 embeddings, shape (26, 1024), dtype float32
